# Step 3: Build the first modeling table

The base table contains the outcome (`target`) and time information, but almost no information for predicting default. In this notebook we add a small set of numerical features from `train_static_0`.

The static data is split into two files. They contain different groups of applications, so we first stack them with `concat()`. We then connect the result to the base table using the shared `case_id` column.

In [1]:
from pathlib import Path
import pandas as pd

project_dir = Path(r"D:\Home Credit - Credit Risk Model Stability")
train_dir = project_dir / "data" / "parquet_files" / "train"

base_path = train_dir / "train_base.parquet"
static_paths = [
    train_dir / "train_static_0_0.parquet",
    train_dir / "train_static_0_1.parquet",
]

assert base_path.exists(), f"File not found: {base_path}"
assert all(path.exists() for path in static_paths), "A static file is missing."
print("All required files were found.")

All required files were found.


## 3.1 Select a small set of features

The static table has 168 columns. Loading every column is unnecessary for our first exercise, so we select ten understandable numerical features. Parquet can read only these columns, which saves memory.

Feature names look unusual because they are anonymized identifiers. The supplied feature dictionary explains their meaning.

In [2]:
selected_features = [
    "credamount_770A",              # Loan amount or credit-card limit
    "annuity_780A",                 # Monthly annuity amount
    "maininc_215A",                 # Primary income amount
    "applications30d_658L",         # Applications made in the last 30 days
    "numactivecreds_622L",          # Number of active credits
    "totaldebt_9A",                  # Total debt amount
    "avgdbddpdlast24m_3658932P",    # Average days before/past due
    "avgdpdtolclosure24_3658938P",  # Average tolerated days past due
    "clientscnt12m_3712952L",       # Clients sharing the phone number
    "price_1097A",                   # Price of the financed item
]

definitions = pd.read_csv(project_dir / "data" / "feature_definitions.csv")
definitions.loc[
    definitions["Variable"].isin(selected_features),
    ["Variable", "Description"],
].set_index("Variable").reindex(selected_features)

,Description
Variable,
credamount_770A,Loan amount or credit card limit.
annuity_780A,Monthly annuity amount.
maininc_215A,Client's primary income amount.
applications30d_658L,Number of applications made by the client in t...
numactivecreds_622L,Number of active credits.
totaldebt_9A,Total amount of debt.
avgdbddpdlast24m_3658932P,Average days past or before due of payment dur...
avgdpdtolclosure24_3658938P,Average DPD (days past due) with tolerance wit...
clientscnt12m_3712952L,Number of clients that have used the same mobi...


## 3.2 Load and combine the two static files

`pd.concat(..., ignore_index=True)` places the rows from the second file below the rows from the first. This is appropriate because both files have the same columns and contain different `case_id` values.

In [3]:
columns_to_read = ["case_id"] + selected_features

static_parts = [
    pd.read_parquet(path, columns=columns_to_read)
    for path in static_paths
]

for path, part in zip(static_paths, static_parts):
    print(f"{path.name}: {len(part):,} rows")

static_data = pd.concat(static_parts, ignore_index=True)
del static_parts

print(f"Combined static table: {static_data.shape[0]:,} rows and {static_data.shape[1]} columns")
static_data.head()

train_static_0_0.parquet: 1,003,757 rows
train_static_0_1.parquet: 522,902 rows
Combined static table: 1,526,659 rows and 11 columns


,case_id,credamount_770A,annuity_780A,maininc_215A,applications30d_658L,numactivecreds_622L,totaldebt_9A,avgdbddpdlast24m_3658932P,avgdpdtolclosure24_3658938P,clientscnt12m_3712952L,price_1097A
0,0,30000.0,1917.6,NaN,0.0,0.0,0.0,NaN,NaN,0.0,NaN
1,1,19999.8,3134.0,NaN,0.0,0.0,0.0,NaN,NaN,0.0,NaN
2,2,78000.0,4937.0,NaN,0.0,0.0,0.0,NaN,NaN,0.0,NaN
3,3,40000.0,4643.6,NaN,1.0,0.0,0.0,NaN,NaN,0.0,NaN
4,4,44000.0,3390.2,NaN,1.0,0.0,0.0,NaN,NaN,0.0,NaN


## 3.3 Check the join key

Before joining, verify that each `case_id` occurs once. A duplicate ID could unexpectedly multiply rows during a join.

In [4]:
duplicate_static_ids = static_data["case_id"].duplicated().sum()
print(f"Duplicate case_id values in static data: {duplicate_static_ids:,}")
assert duplicate_static_ids == 0, "Stop: duplicate IDs would make this join unsafe."

Duplicate case_id values in static data: 0


## 3.4 Join the features to the base table

A **left join** keeps every application from the base table and adds matching static features. `validate="one_to_one"` asks pandas to confirm that neither side has duplicate IDs.

In [5]:
train_base = pd.read_parquet(base_path)
train_base["date_decision"] = pd.to_datetime(
    train_base["date_decision"], format="%Y-%m-%d"
)

modeling_data = train_base.merge(
    static_data,
    on="case_id",
    how="left",
    validate="one_to_one",
)

assert len(modeling_data) == len(train_base)
print(f"Modeling table: {modeling_data.shape[0]:,} rows and {modeling_data.shape[1]} columns")
modeling_data.head()

Modeling table: 1,526,659 rows and 15 columns


,case_id,date_decision,MONTH,WEEK_NUM,target,credamount_770A,annuity_780A,maininc_215A,applications30d_658L,numactivecreds_622L,totaldebt_9A,avgdbddpdlast24m_3658932P,avgdpdtolclosure24_3658938P,clientscnt12m_3712952L,price_1097A
0,0,2019-01-03,201901,0,0,30000.0,1917.6,NaN,0.0,0.0,0.0,NaN,NaN,0.0,NaN
1,1,2019-01-03,201901,0,0,19999.8,3134.0,NaN,0.0,0.0,0.0,NaN,NaN,0.0,NaN
2,2,2019-01-04,201901,0,0,78000.0,4937.0,NaN,0.0,0.0,0.0,NaN,NaN,0.0,NaN
3,3,2019-01-03,201901,0,0,40000.0,4643.6,NaN,1.0,0.0,0.0,NaN,NaN,0.0,NaN
4,4,2019-01-04,201901,0,1,44000.0,3390.2,NaN,1.0,0.0,0.0,NaN,NaN,0.0,NaN


## 3.5 Examine missing feature values

Unlike the base table, predictor columns can contain missing values. Missing data does not automatically mean the row is bad. For example, an applicant may have no previous credit history or a field may not have been collected.

For now, we only measure the missingness. We will decide how to handle it when building the model.

In [6]:
feature_summary = pd.DataFrame({
    "data_type": modeling_data[selected_features].dtypes.astype(str),
    "missing_count": modeling_data[selected_features].isna().sum(),
    "missing_percent": modeling_data[selected_features].isna().mean().mul(100).round(2),
    "unique_values": modeling_data[selected_features].nunique(dropna=True),
}).sort_values("missing_percent", ascending=False)

feature_summary

,data_type,missing_count,missing_percent,unique_values
avgdbddpdlast24m_3658932P,float64,613202,40.17,4524
maininc_215A,float64,511255,33.49,7738
avgdpdtolclosure24_3658938P,float64,467175,30.60,4313
price_1097A,float64,223240,14.62,173566
credamount_770A,float64,0,0.00,183502
annuity_780A,float64,0,0.00,75325
applications30d_658L,float64,0,0.00,25
numactivecreds_622L,float64,0,0.00,8
totaldebt_9A,float64,3,0.00,363903
clientscnt12m_3712952L,float64,0,0.00,89


## 3.6 Create two simple financial ratios

A **feature** is an input given to a model. Feature engineering creates a potentially useful input from existing columns. Here we compare the requested credit and monthly annuity with reported income.

Income values that are missing or not positive produce a missing ratio. We avoid filling them with an arbitrary number at this stage.

In [7]:
valid_income = modeling_data["maininc_215A"].where(
    modeling_data["maininc_215A"] > 0
)

modeling_data["credit_to_income_ratio"] = (
    modeling_data["credamount_770A"] / valid_income
)
modeling_data["annuity_to_income_ratio"] = (
    modeling_data["annuity_780A"] / valid_income
)

ratio_columns = [
    "credamount_770A", "annuity_780A", "maininc_215A",
    "credit_to_income_ratio", "annuity_to_income_ratio",
]
modeling_data[ratio_columns].dropna().head(10)

,credamount_770A,annuity_780A,maininc_215A,credit_to_income_ratio,annuity_to_income_ratio
14,20000.0,2484.8000,50000.0,0.400000,0.049696
20,100000.0,6000.0000,96000.0,1.041667,0.062500
38,30450.0,1827.0000,20000.0,1.522500,0.091350
69,40000.0,2400.0000,14000.0,2.857143,0.171429
103,36540.0,2192.4001,54904.0,0.665525,0.039932
105,64000.0,5061.0000,65000.0,0.984615,0.077862
127,100000.0,5103.0000,9400.0,10.638298,0.542872
143,16000.0,1975.8000,92000.0,0.173913,0.021476
144,10000.0,600.0000,130000.0,0.076923,0.004615
147,100000.0,5408.2000,56000.0,1.785714,0.096575


## 3.7 Final checks

The final table should still have one row per application. `case_id` remains the join key, `target` is the outcome, and the selected and engineered columns are candidate model inputs.

In [8]:
print(f"Rows: {len(modeling_data):,}")
print(f"Columns: {modeling_data.shape[1]}")
print(f"Duplicate application IDs: {modeling_data['case_id'].duplicated().sum():,}")
print(f"Approximate memory used: {modeling_data.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
print("Candidate model features:")
print(selected_features + ["credit_to_income_ratio", "annuity_to_income_ratio"])

Rows: 1,526,659
Columns: 17
Duplicate application IDs: 0
Approximate memory used: 198.0 MB
Candidate model features:
['credamount_770A', 'annuity_780A', 'maininc_215A', 'applications30d_658L', 'numactivecreds_622L', 'totaldebt_9A', 'avgdbddpdlast24m_3658932P', 'avgdpdtolclosure24_3658938P', 'clientscnt12m_3712952L', 'price_1097A', 'credit_to_income_ratio', 'annuity_to_income_ratio']


### What you learned

1. Parquet can load only selected columns, reducing memory use.
2. Files with the same schema can be stacked with `concat()`.
3. Related tables can be connected with a shared key using `merge()`.
4. Join keys must be checked for duplicates before merging.
5. Predictor fields often contain missing values, and missingness may itself carry information.
6. Existing columns can be combined into new features such as financial ratios.

The next notebook will split applications by time, train a simple baseline classifier, and evaluate it using ROC AUC rather than ordinary accuracy.